# 01 · Análisis exploratorio de Superstore

El objetivo es entender los importes por línea de pedido y determinar qué permite observar su agregación por mes. Se usa el CSV versionado del proyecto y el mismo contrato que usa el entrenamiento. La unidad monetaria de `Sales` sigue por confirmar.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from superstore_forecasting.config import RAW_PATH
from superstore_forecasting.data import load_orders
from superstore_forecasting.features import aggregate_monthly

root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src" / "superstore_forecasting").is_dir()
)
dataset_path = root / RAW_PATH
if not dataset_path.is_file():
    raise FileNotFoundError(
        "Desde la raíz, ejecuta: uv run --locked python -m scripts.download_data"
    )

plt.style.use("seaborn-v0_8-whitegrid")
orders = load_orders(dataset_path)
monthly = aggregate_monthly(orders)

## Estructura y estadísticas

`load_orders` conserva las filas y valida las columnas requeridas, el formato día/mes/año y los importes finitos y positivos. Conserva como texto los campos auxiliares y convierte en nulos los códigos postales vacíos. La vista inicial muestra una selección de columnas sin nombres de clientes.

In [ ]:
print(f"Dimensiones: {orders.shape[0]:,} filas × {orders.shape[1]} columnas")
display(orders.dtypes.astype(str).rename("Tipo").to_frame())
display(orders[["Order Date", "Sales", "Category", "Region", "Segment"]].head())
display(orders["Sales"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]).to_frame())

## Calidad, faltantes y duplicados

Un pedido puede contener varias líneas. Se cuentan por separado las filas idénticas y las líneas con un identificador de pedido repetido. El contrato conserva ambos casos y no imputa códigos postales ni ventas.

In [ ]:
missing = (
    (orders.isna() | orders.eq("").fillna(False)).sum().sort_values(ascending=False)
)
display(missing.rename("Valores faltantes").to_frame())
display(
    pd.Series(
        {
            "Filas idénticas adicionales": orders.duplicated().sum(),
            "Líneas adicionales de un mismo pedido": orders["Order ID"]
            .duplicated()
            .sum(),
            "Meses consecutivos con registros": len(monthly),
            "Primer pedido": orders["Order Date"].min().date(),
            "Último pedido": orders["Order Date"].max().date(),
        }
    ).to_frame("Resultado")
)

## Distribución e importes extremos

El histograma usa todo el rango. El percentil 99 es una referencia descriptiva para investigar la cola, sin convertirse en una regla de filtrado.

In [ ]:
sales = orders["Sales"]
p99 = sales.quantile(0.99)
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
ax.hist(sales, bins=55, color="#267b82", alpha=0.9)
ax.axvline(p99, color="#bf6335", linestyle="--", label=f"P99 = {p99:,.2f}")
ax.set(
    title="Distribución del importe por línea de pedido",
    xlabel="Importe (unidad de Sales)",
    ylabel="Número de líneas",
    yscale="log",
)
ax.legend()
plt.show()
display(
    Markdown(
        f"La mediana es **{sales.median():,.2f}** y el máximo **{sales.max():,.2f}**. "
        f"Hay **{sales.gt(p99).sum()}** líneas por encima del percentil 99. "
        "La escala logarítmica del eje vertical permite ver las frecuencias bajas de la cola."
    )
)

## Ventas acumuladas por categoría y región

Los totales corresponden a todas las líneas del periodo observado. Reflejan el volumen registrado por grupo y no incluyen información de margen.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for ax, group, label in zip(
    axes, ("Category", "Region"), ("Categoría", "Región"), strict=True
):
    totals = orders.groupby(group)["Sales"].sum().sort_values()
    totals.plot.barh(ax=ax, color="#267b82")
    ax.set(
        title=f"Ventas acumuladas por {label.lower()}",
        xlabel="Ventas (unidad de Sales)",
        ylabel=label,
    )
    display(Markdown(f"Mayor total por {label.lower()}: **{totals.idxmax()}**."))
plt.show()

## Evolución, tendencia y crecimiento interanual

La media móvil de 12 meses suaviza la estacionalidad y se muestra solo para explorar la serie. La variación interanual compara cada mes con el mismo mes del año anterior. Los primeros 12 meses carecen de comparación.

In [ ]:
timeline = monthly["sales"].set_axis(monthly.index.to_timestamp())
yoy = timeline.pct_change(periods=12, fill_method=None).mul(100)
fig, (ax_sales, ax_yoy) = plt.subplots(
    2, 1, figsize=(11, 7), sharex=True, layout="constrained"
)
timeline.plot(ax=ax_sales, color="#267b82", label="Ventas observadas")
timeline.rolling(12).mean().plot(
    ax=ax_sales, color="#bf6335", label="Media móvil de 12 meses"
)
ax_sales.set(title="Evolución mensual de ventas", ylabel="Ventas (unidad de Sales)")
ax_sales.legend()
yoy.plot(ax=ax_yoy, color="#4d65a8")
ax_yoy.axhline(0, color="0.4", linewidth=1)
ax_yoy.set(
    title="Variación respecto al mismo mes del año anterior",
    ylabel="Variación (%)",
    xlabel="Mes",
)
plt.show()

## Estacionalidad y comparación por año

In [ ]:
seasonal = monthly.assign(year=monthly.index.year, month=monthly.index.month)
year_by_month = seasonal.pivot(index="month", columns="year", values="sales")
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
year_by_month.plot(ax=ax, marker="o")
ax.set(
    title="Perfil mensual de ventas por año",
    xlabel="Mes del año",
    ylabel="Ventas (unidad de Sales)",
    xticks=range(1, 13),
    xticklabels="Ene Feb Mar Abr May Jun Jul Ago Sep Oct Nov Dic".split(),
)
ax.legend(title="Año")
plt.show()
annual = year_by_month.sum().rename("Ventas observadas")
display(annual.to_frame().assign(variacion_pct=annual.pct_change().mul(100)))
peak = year_by_month.mean(axis=1).idxmax()
display(Markdown(f"El mayor promedio mensual observado corresponde al mes **{peak}**."))

## Conclusiones y límites

La comparación por mes permite plantear un baseline anual. Hay pocos ciclos para separar estacionalidad, tendencia y eventos puntuales. Tener registros en todos los meses no demuestra que cada cierre sea completo. La media móvil se usa únicamente en este EDA. Los totales agregados tampoco permiten atribuir cambios a promociones, precios o composición de productos.